# 项目架构与进展分析笔记

## 动机

用户要求重述当前项目做了哪些事情。我们需要梳理项目结构，以明确现有代码实现了哪些对比方法（baselines）和 `vcaan` 模型，从而为将来开发新改进的模型（Improved Model）做好准备。

## 1. 现有项目总结

从项目结构和代码来看，这是一个专门为了**多变量时间序列缺失值插补 (Multivariate Time Series Imputation)** 搭建的标准化实验框架，所有的实验都可以通过 `main.py` 流水线式运行。

目前已经完成并可用的工作包括：
1. **数据集准备流水线**（`src/data`）：支持构建不同缺失模式（`mcar`, `seq`连续缺失, `scm`空间相关的缺失）和通过指定的缺失率（10%, 30%, 50%）自动生成对应的掩码(Mask)。支持时间序列按年按月进行固定的 Train/Val/Test 划分。最后也实现了特征的时间窗切分以送入深度模型。
2. **完整的基线对比方法库**（`src/models/baselines.py`）：
   - **PyPOTS（深度学习类）**：包含了 `saits`（基于 Self-Attention）, `grud`（基于 RNN）, `usgan`（基于 GAN）, `itransformer`。
   - **机器学习方法**：包含了 `knn`, `mice`（代码中还实现了基于分块 `Chunked` 的方法以避免长序列预测过慢的问题）。
   - **简易统计方法**：如 `locf`（最近观测值结转）。
3. **前期论文模型：VCAAN**（`src/models/vcaan.py`）：用户提到的前期已发表论文的模型。在当前代码中，实现为一种迭代优化的时序空间模型：它**先使用 LOCF 进行预填充（Warm Start）**，接着提取时空相关性特征矩阵，然后利用**线性回归迭代式**优化缺失位置的取值，直至达到收敛阈值。它作为一个baseline选项集成在了这个框架中。
4. **标准的评估与自动化日志系统**：
   - **评价指标（`src/evaluate.py`）**：严格计算了只有真实**缺失位置（missing positions）上的 RMSE**。
   - **日志与可视化**：每次实验记录详细 config 与指标表到 `logs/` 目录；`scripts/plot_baseline_compare.py` 支持将不同模型的结果绘制为对比柱状图。

## 2. 下一步改进动机

项目的脚手架（数据划分、缺失构造、训练测试流程、统一评测标准）已经非常健康且标准化。用户的动机是：**如果想要在这个标准化跑道上，实现一个新的、改进的插补模型**。这就意味着我们在接下来的步骤中，架构上只需要：
1. 在 `src/models/` 目录下构思并新建新模型的核心代码逻辑。
2. 将新模型注册和桥接到 `src/models/baselines.py` 和 `main.py` 的解析参数中。
3. 新模型可直接通过 `main.py` 复用已有的数据集和评价体系，无缝地和 VCAAN 及其他已有基线同台比对了。